In [29]:
from pathlib import Path
import pandas as pd

RACINE = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / ".git").exists())
DOSSIER = RACINE / "data" / "raw" / "prix"
BENCH = RACINE / "data" / "raw" / "benchmarks"

In [30]:
CONTROLE = RACINE / "data" / "processed" / "controle_prix.csv"

seances = set(pd.read_csv(RACINE / "data" / "raw" / "calendrier_bourse.csv")["date"].astype(str))
debut_cal, fin_cal = min(seances), max(seances)

In [31]:
anomalies = []

for fichier in sorted(list(DOSSIER.glob("*.csv")) + list(BENCH.glob("*.csv"))):
    dates = pd.read_csv(fichier, usecols=["date"])["date"].str[:10]
    debut, fin = dates.iloc[0], dates.iloc[-1]

    for j in dates[dates.duplicated()].unique():
        anomalies.append({"fichier": fichier.name, "test": "date en double",
                          "date": j, "detail": ""})

    avant = int((dates < debut_cal).sum())
    if avant:
        anomalies.append({"fichier": fichier.name, "test": "anteriorite au calendrier",
                          "date": debut, "detail": f"{avant} seances avant {debut_cal}"})

    bas, haut = max(debut, debut_cal), min(fin, fin_cal)
    couvertes = set(dates[(dates >= bas) & (dates <= haut)])
    attendues = {j for j in seances if bas <= j <= haut}

    for j in sorted(attendues - couvertes):
        anomalies.append({"fichier": fichier.name, "test": "seance absente",
                          "date": j, "detail": ""})
    for j in sorted(couvertes - seances):
        anomalies.append({"fichier": fichier.name, "test": "date hors calendrier",
                          "date": j, "detail": ""})

controle = pd.DataFrame(anomalies)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(controle), "anomalies")
if len(controle):
    print(controle.test.value_counts().to_string())
    print()
    print(controle[controle.test == "seance absente"].fichier.value_counts().head(10).to_string())

74 anomalies
test
anteriorite au calendrier    60
seance absente               14

fichier
SPXEW.csv    14


In [32]:
SEUIL = 0.30
TESTS = ["variation quotidienne extreme", "barre incoherente",
         "prix nul ou negatif", "valeur manquante"]
COLS = ["Open", "High", "Low", "Close", "Adj Close"]

In [33]:
anomalies = []

for fichier in sorted(list(DOSSIER.glob("*.csv")) + list(BENCH.glob("*.csv"))):
    d = pd.read_csv(fichier, usecols=["date"] + COLS)
    d["j"] = d["date"].str[:10]

    var = d["Adj Close"].pct_change(fill_method=None)
    for i in var[var.abs() > SEUIL].index:
        anomalies.append({"fichier": fichier.name, "test": "variation quotidienne extreme",
                          "date": d.j[i], "detail": f"{var[i] * 100:+.1f} %"})

    incoherent = ((d.Low > d.High) | (d.Open < d.Low) | (d.Open > d.High)
                  | (d.Close < d.Low) | (d.Close > d.High))
    for i in d.index[incoherent]:
        r = d.loc[i]
        anomalies.append({"fichier": fichier.name, "test": "barre incoherente", "date": r.j,
                          "detail": f"O={r.Open:.2f} H={r.High:.2f} L={r.Low:.2f} C={r.Close:.2f}"})

    for i in d.index[(d[COLS] <= 0).any(axis=1)]:
        anomalies.append({"fichier": fichier.name, "test": "prix nul ou negatif",
                          "date": d.j[i], "detail": ""})

    for i in d.index[d[COLS].isna().any(axis=1)]:
        anomalies.append({"fichier": fichier.name, "test": "valeur manquante",
                          "date": d.j[i], "detail": ""})

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(anomalies), "nouvelles anomalies |", len(controle), "au total")
print(pd.DataFrame(anomalies).test.value_counts().to_string())

129 nouvelles anomalies | 203 au total
test
variation quotidienne extreme    127
barre incoherente                  1
valeur manquante                   1


In [34]:
TOL = 1e-4
TESTS_15 = ["ajustement incoherent"]

In [35]:
anomalies, ecarts = [], []

for fichier in sorted(list(DOSSIER.glob("*.csv")) + list(BENCH.glob("*.csv"))):
    d = pd.read_csv(fichier, usecols=["date", "Close", "Adj Close", "Dividends", "Stock Splits"])
    d["j"] = d["date"].str[:10]

    r_ajuste = d["Adj Close"].pct_change(fill_method=None)
    r_additif = (d["Close"] + d["Dividends"]) / d["Close"].shift() - 1
    r_multiplicatif = d["Close"] / (d["Close"].shift() - d["Dividends"]) - 1

    e_add = (r_ajuste - r_additif).abs()
    e_mul = (r_ajuste - r_multiplicatif).abs()
    ecarts.append({"fichier": fichier.name, "additif": e_add.max(),
                   "multiplicatif": e_mul.max()})

    for i in e_mul[e_mul > TOL].index:
        anomalies.append({"fichier": fichier.name, "test": "ajustement incoherent",
                          "date": d.j[i], "detail": f"ecart {e_mul[i]:.2e}"})

ecarts = pd.DataFrame(ecarts)

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_15)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print("ecart maximal, tous fichiers confondus")
print(ecarts[["additif", "multiplicatif"]].max().to_string())
print()
print(ecarts.sort_values("additif", ascending=False).head(8).to_string(index=False))
print()
print(len(anomalies), "ajustements incoherents |", len(controle), "au total")

ecart maximal, tous fichiers confondus
additif          0.081221
multiplicatif    0.000043

 fichier  additif  multiplicatif
 JCI.csv 0.081221   1.438157e-06
GNRC.csv 0.037420   1.559874e-07
 ETN.csv 0.036709   1.780154e-06
 BKR.csv 0.022124   1.620049e-06
 CNP.csv 0.017817   2.569881e-06
 CRH.csv 0.010801   8.676735e-07
 SLB.csv 0.009759   1.517299e-06
HUBB.csv 0.006409   4.827455e-06

0 ajustements incoherents | 203 au total


In [36]:
TESTS_16 = ["division non confirmee"]
DEBUT_SEC = "2010-01-01"
TOL_SPLIT = 0.02

In [37]:
sp500 = pd.read_csv(RACINE / "data" / "raw" / "sp500_constituents.csv", dtype=str)
symbole = dict(zip(sp500.CIK, sp500.Symbol.str.replace(".", "-", regex=False)))

actions = pd.read_csv(RACINE / "data" / "raw" / "actions_en_circulation.csv", dtype={"cik": str})
actions = actions[actions.notion == "actions"].copy()
actions["symbole"] = actions.cik.map(symbole)
actions = actions.dropna(subset=["symbole"])
actions = actions.sort_values(["depose_le", "fin", "valeur"], kind="stable")

In [38]:
anomalies, mesures = [], []

for fichier in sorted(DOSSIER.glob("*.csv")):
    d = pd.read_csv(fichier, usecols=["date", "Stock Splits"]).rename(
        columns={"Stock Splits": "division"})
    d["j"] = d["date"].str[:10]
    divisions = d[(d.division > 0) & (d.j >= DEBUT_SEC)]
    if divisions.empty:
        continue

    serie = actions[actions.symbole == fichier.stem]

    for r in divisions.itertuples():
        avant = serie[serie.depose_le < r.j].tail(1)
        apres = serie[serie.depose_le > r.j].head(1)
        if avant.empty or apres.empty:
            mesures.append({"symbole": fichier.stem, "date": r.j, "annonce": r.division,
                            "mesure": None, "ecart": None})
            continue

        mesure = apres.valeur.iloc[0] / avant.valeur.iloc[0]
        ecart = abs(mesure / r.division - 1)
        mesures.append({"symbole": fichier.stem, "date": r.j, "annonce": r.division,
                        "mesure": mesure, "ecart": ecart})

        if ecart > TOL_SPLIT:
            anomalies.append({"fichier": fichier.name, "test": "division non confirmee",
                              "date": r.j,
                              "detail": f"annonce {r.division:g}, mesure {mesure:.3f}"})

mesures = pd.DataFrame(mesures)

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_16)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(mesures), "divisions depuis", DEBUT_SEC)
print(int(mesures.mesure.isna().sum()), "sans encadrement SEC")
print(len(anomalies), "non confirmees |", len(controle), "au total")
print()
print(mesures.dropna(subset=["ecart"]).sort_values("ecart", ascending=False)
      .head(24).to_string(index=False))

56 divisions depuis 2010-01-01
9 sans encadrement SEC
20 non confirmees | 223 au total

symbole       date   annonce    mesure    ecart
    JCI 2016-09-06  0.955000  2.197711 1.301268
  BRK-B 2010-01-21 50.000000  1.044357 0.979113
     NI 2015-07-02  2.545000  1.001517 0.606477
    DUK 2012-07-03  0.333333  0.526211 0.578632
    JCI 2012-10-01  2.011668  1.012704 0.496585
      O 2021-11-15  1.032000  1.462894 0.417533
   FLEX 2024-01-03  1.327000  0.968512 0.270149
    HPE 2017-04-03  1.334800  0.990907 0.257637
    WDC 2025-02-24  1.323000  1.003032 0.241851
    HPE 2017-09-01  1.289000  0.985897 0.235146
     TT 2013-12-02  1.252000  0.965112 0.229144
    DOV 2018-05-09  1.238000  0.954912 0.228666
     TT 2020-03-02  1.289000  1.003385 0.221579
    DOV 2014-03-03  1.205000  0.979243 0.187350
    WMB 2012-01-03  1.226693  1.004701 0.180968
    MMM 2024-04-01  1.196000  1.001196 0.162879
    DTE 2021-07-01  1.175000  1.000125 0.148830
    IRM 2014-09-26  1.082000  1.003054 0.072963


In [39]:
TESTS_17 = ["reference de cotation absente", "premiere cotation discordante",
            "historique tronque"]

composants = pd.read_csv(RACINE / "data" / "raw" / "sp500_constituents.csv", dtype=str)
entree = dict(zip(composants.Symbol.str.replace(".", "-", regex=False),
                  composants["Date added"]))

reference = pd.read_csv(RACINE / "data" / "raw" / "premieres_cotations.csv")
premiere = dict(zip(reference.ticker, reference.premiere_cotation))

In [40]:
anomalies, debuts = [], []

for fichier in sorted(DOSSIER.glob("*.csv")):
    debut = pd.read_csv(fichier, usecols=["date"], nrows=1).date.iloc[0][:10]
    ref = premiere.get(fichier.stem)
    ajout = entree.get(fichier.stem)
    debuts.append({"symbole": fichier.stem, "prix": debut, "reference": ref, "indice": ajout})

    if ref is None:
        anomalies.append({"fichier": fichier.name, "test": "reference de cotation absente",
                          "date": debut, "detail": ""})
    elif ref != debut:
        anomalies.append({"fichier": fichier.name, "test": "premiere cotation discordante",
                          "date": debut, "detail": f"reference {ref}"})

    if ajout is not None and ajout < debut:
        anomalies.append({"fichier": fichier.name, "test": "historique tronque",
                          "date": debut, "detail": f"entree dans l'indice le {ajout}"})

debuts = pd.DataFrame(debuts)

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_17)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(anomalies), "anomalies |", len(controle), "au total")
print(pd.DataFrame(anomalies).test.value_counts().to_string())
print()
print("dates de debut partagees par plusieurs titres")
compte = debuts.prix.value_counts()
print(compte[compte > 1].to_string())

57 anomalies | 280 au total
test
reference de cotation absente    37
historique tronque               20

dates de debut partagees par plusieurs titres
prix
1980-03-17    12
1973-02-21    10
1962-01-02     8
1981-12-31     4
1972-06-01     4
2004-08-19     2


In [41]:
import time
import yfinance as yf

META = RACINE / "data" / "raw" / "metadonnees_titres.csv"

fichiers = sorted(DOSSIER.glob("*.csv")) + sorted(BENCH.glob("*.csv"))
lignes, echecs = [], []

for fichier in fichiers:
    declare = pd.read_csv(fichier, usecols=["symbole"], nrows=1).symbole.iloc[0]
    interroge = declare.replace(".", "-")

    try:
        m = yf.Ticker(interroge).history_metadata
    except Exception as erreur:
        echecs.append((interroge, type(erreur).__name__))
        continue

    ts = m.get("firstTradeDate")
    if not ts:
        echecs.append((interroge, "aucune premiere transaction declaree"))
        continue

    lignes.append({
        "fichier": fichier.name,
        "symbole": interroge,
        "symbole_declare": declare,
        "devise": m.get("currency"),
        "place": m.get("fullExchangeName"),
        "type": m.get("instrumentType"),
        "fuseau": m.get("timezone"),
        "nom": m.get("longName") or m.get("shortName"),
        "premiere_transaction": str(pd.to_datetime(ts, unit="s", utc=True).date()),
    })
    time.sleep(0.2)

meta = pd.DataFrame(lignes)
meta["collecte_le"] = pd.Timestamp.utcnow().strftime("%Y-%m-%d")
meta.to_csv(META, index=False, encoding="utf-8")

print(len(meta), "titres decrits |", len(echecs), "echecs")
if echecs:
    print(echecs)
print()
print(meta.devise.value_counts().to_string())
print(meta.type.value_counts().to_string())
print(meta.place.value_counts().to_string())
print()
divergents = meta[meta.symbole != meta.symbole_declare]
print(len(divergents), "fichiers ou le symbole declare differe de celui interroge")
if len(divergents):
    print(divergents[["fichier", "symbole", "symbole_declare"]].to_string(index=False))

119 titres decrits | 0 echecs

devise
USD    119
type
EQUITY    114
INDEX       3
ETF         2
place
NYSE        72
NasdaqGS    43
SNP          2
NYSEArca     2

1 fichiers ou le symbole declare differe de celui interroge
  fichier symbole symbole_declare
BRK-B.csv   BRK-B           BRK.B


In [42]:
TESTS_18 = ["metadonnees absentes", "devise non usd", "fuseau inattendu", "type inattendu",
            "decalage horaire inattendu", "premiere transaction discordante"]

FUSEAUX = {"EST", "EDT"}
DECALAGES = {"-05:00", "-04:00"}

meta = pd.read_csv(RACINE / "data" / "raw" / "metadonnees_titres.csv")
attendu = {r.fichier: r.premiere_transaction for r in meta.itertuples()
           if isinstance(r.premiere_transaction, str)}
decrits = set(meta.fichier)
anomalies = []

for r in meta.itertuples():
    if r.devise != "USD":
        anomalies.append({"fichier": r.fichier, "test": "devise non usd",
                          "date": "", "detail": str(r.devise)})
    if r.fuseau not in FUSEAUX:
        anomalies.append({"fichier": r.fichier, "test": "fuseau inattendu",
                          "date": "", "detail": str(r.fuseau)})

    attendus = {"EQUITY"} if (DOSSIER / r.fichier).exists() else {"ETF", "INDEX"}
    if r.type not in attendus:
        anomalies.append({"fichier": r.fichier, "test": "type inattendu",
                          "date": "", "detail": str(r.type)})

for fichier in fichiers:
    if fichier.name not in decrits:
        anomalies.append({"fichier": fichier.name, "test": "metadonnees absentes",
                          "date": "", "detail": ""})

    d = pd.read_csv(fichier, usecols=["date"])
    for x in sorted(set(d.date.str[-6:]) - DECALAGES):
        anomalies.append({"fichier": fichier.name, "test": "decalage horaire inattendu",
                          "date": "", "detail": x})

    debut = d.date.iloc[0][:10]
    declaree = attendu.get(fichier.name)
    if declaree and declaree != debut:
        anomalies.append({"fichier": fichier.name, "test": "premiere transaction discordante",
                          "date": debut, "detail": f"declaree {declaree}"})

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_18)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(anomalies), "anomalies |", len(controle), "au total")
if anomalies:
    print(pd.DataFrame(anomalies).test.value_counts().to_string())

0 anomalies | 280 au total


In [43]:
TESTS_19 = ["fin de serie anticipee", "denomination divergente"]

import re

def normaliser(nom):
    n = str(nom).lower()
    for mot in [" incorporated", " corporation", " companies", " company", " holdings",
                " group", " inc", " corp", " plc", " ltd", " the", " co",
                " & ", " and ", ".", ",", "'", "-"]:
        n = n.replace(mot, " ")
    return re.sub(r"\s+", " ", n).strip()

calendrier = pd.read_csv(RACINE / "data" / "raw" / "calendrier_bourse.csv")["date"].astype(str)
derniere = calendrier.max()

meta = pd.read_csv(RACINE / "data" / "raw" / "metadonnees_titres.csv")
composants = pd.read_csv(RACINE / "data" / "raw" / "sp500_constituents.csv", dtype=str)
nom_indice = dict(zip(composants.Symbol.str.replace(".", "-", regex=False), composants.Security))

anomalies = []

for fichier in fichiers:
    fin = pd.read_csv(fichier, usecols=["date"]).date.iloc[-1][:10]
    if fin < derniere:
        anomalies.append({"fichier": fichier.name, "test": "fin de serie anticipee",
                          "date": fin, "detail": f"derniere seance {derniere}"})

for r in meta.itertuples():
    reference = nom_indice.get(r.symbole)
    if reference is None:
        continue
    a, b = normaliser(r.nom), normaliser(reference)
    if not (a.startswith(b[:8]) or b.startswith(a[:8])):
        anomalies.append({"fichier": r.fichier, "test": "denomination divergente",
                          "date": "", "detail": f"{r.nom} contre {reference}"})

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_19)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(anomalies), "anomalies |", len(controle), "au total")
if anomalies:
    print(pd.DataFrame(anomalies).test.value_counts().to_string())
    print()
    print(pd.DataFrame(anomalies)[["fichier", "detail"]].to_string(index=False))

6 anomalies | 286 au total
test
denomination divergente    6

 fichier                                                 detail
 AES.csv             The AES Corporation contre AES Corporation
 IBM.csv International Business Machines Corporation contre IBM
 SLB.csv                           SLB N.V. contre Schlumberger
SMCI.csv           Super Micro Computer, Inc. contre Supermicro
  SO.csv           The Southern Company contre Southern Company
 WMB.csv The Williams Companies, Inc. contre Williams Companies


In [44]:
TESTS_19B = ["seance sans transaction", "prix fige"]

anomalies = []
resume = []

for fichier in sorted(DOSSIER.glob("*.csv")):
    d = pd.read_csv(fichier, usecols=["date", "Open", "High", "Low", "Close", "Volume"])
    d["j"] = d["date"].str[:10]

    sans_volume = d.Volume == 0
    fige = (sans_volume
            & (d.Open == d.High) & (d.High == d.Low) & (d.Low == d.Close)
            & (d.Close == d.Close.shift()))

    if sans_volume.any():
        resume.append({"symbole": fichier.stem, "lignes": len(d),
                       "sans_volume": int(sans_volume.sum()), "fige": int(fige.sum()),
                       "part": int(sans_volume.sum()) / len(d)})

    for i in d.index[sans_volume & ~fige]:
        anomalies.append({"fichier": fichier.name, "test": "seance sans transaction",
                          "date": d.j[i], "detail": ""})
    for i in d.index[fige]:
        anomalies.append({"fichier": fichier.name, "test": "prix fige",
                          "date": d.j[i], "detail": f"{d.Close[i]:.4f}"})

resume = pd.DataFrame(resume).sort_values("sans_volume", ascending=False)

ancien = pd.read_csv(CONTROLE)
ancien = ancien[~ancien.test.isin(TESTS_19B)]
controle = pd.concat([ancien, pd.DataFrame(anomalies)], ignore_index=True)
controle.to_csv(CONTROLE, index=False, encoding="utf-8")

print(len(anomalies), "anomalies |", len(controle), "au total")
print(pd.DataFrame(anomalies).test.value_counts().to_string())
print()
print(resume.head(12).to_string(index=False, formatters={"part": "{:.1%}".format}))

8388 anomalies | 8674 au total
test
prix fige                  8273
seance sans transaction     115

symbole  lignes  sans_volume  fige  part
   HUBB   13557         5561  5510 41.0%
    CRH    9356         1718  1718 18.4%
   COHR    9805          355   355  3.6%
    TPL   11713          256   231  2.2%
    EME    7967          108   108  1.4%
    VMC   13499           86    86  0.6%
   SWKS   10580           72    72  0.7%
    TER   13499           63    63  0.5%
    HWM    2474           35     0  1.4%
    IEX    9384           17    17  0.2%
   KLAC   11570           13    13  0.1%
    VRT    2034           12    12  0.6%


In [45]:
import yfinance as yf

SYMBOLE = "CMS"
CIBLE = DOSSIER / f"{SYMBOLE}.csv"

if CIBLE.exists():
    raise FileExistsError(f"{CIBLE} existe deja, collecte refusee")

h = yf.Ticker(SYMBOLE).history(period="max", auto_adjust=False, actions=True)
if h.empty:
    raise ValueError(f"aucune donnee pour {SYMBOLE}")

h.index.name = "date"
h["symbole"] = SYMBOLE
h = h[["Open", "High", "Low", "Close", "Adj Close", "Volume",
       "Dividends", "Stock Splits", "symbole"]]
h.to_csv(CIBLE, encoding="utf-8")

print(len(h), "lignes ecrites dans", CIBLE)
print(h.index[0].date(), "->", h.index[-1].date())

FileExistsError: C:\Users\josue\Documents\risk-modeling-project\data\raw\prix\CMS.csv existe deja, collecte refusee